# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [1]:
from crewai import Agent, Task, Crew, LLM
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [2]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

### START CODE HERE ###

# Create the EXASearchTool instance
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance
scrape_website_tool = ScrapeWebsiteTool()

### END CODE HERE ###

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In [3]:
# define the research planner agent
research_planner = Agent(
    role="Research Planner",
    goal="Analyze queries and break them down into smaller, specific research topics.",
    backstory=(
         "You are a research strategist who excels at breaking down complex questions "
         "into manageable research components. You identify what needs to be researched "
         "and create clear research objectives."
    ),
    verbose=True # set to True to see detailed agent actions
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [4]:

researcher = Agent(
    role="Internet Researcher",
    goal="Research thoroughly all assigned topics",
    ### START CODE HERE ###
    backstory=(
        "You are a skilled researcher with experience in online investigation "
        "and data collection. You know how to find reliable sources, extract relevant information, "
        " and always verify facts across multiple sources to avoid misinformation or hallucinations."
        "You nver invent facts and always trace information to its origin."
    ),
    # add the 2 tool instances you created
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True
)

fact_checker = Agent(
    role="Fact Checker",
    goal=(
        "Verify data for accuracy, identify inconsistencies, "
        "and flag potential misinformation"
    ),
    ### START CODE HERE ###
    backstory=( 
        "You are a quality assurance specialist with expertise in fact-checking " 
        "and identifying misinformation and hallucinations. You cross-reference information, "
        "check for hallucinated or invented content and require that all facts be supported " 
        "by evidence."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    ### END CODE HERE ###
    verbose=True
)

report_writer = Agent(
    role="Report Writer",
    goal="Write clear, concise, and well-structured reports based on gathered information",
    ### START CODE HERE ###
    backstory=( 
         "You are an expert writer who specializes in creating clear, well-structured " 
         "research rerports. You synthesize complex information into readable formats and "
         "always include proper citatations and sources."
    ),
    ### END CODE HERE ###
    verbose=True
)
        

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

In [5]:
# define the create research plan task
create_research_plan_task = Task(
    description=(
        "Based on the user's query, break it down into specific topics and key questions, "
        "and create a focused research plan."
        "The user's query is: {user_query}"
    ),
    expected_output=(
        "A research plan with main research topics to investigate, "
        "key questions for each topic, and success criteria for the research."
        ),
    agent=research_planner,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [6]:
# define the gather research data task
gather_research_data_task = Task(
    description=(
        "Using the research plan, collect information on all identified topics. "
        "Cite all sources used."
    ),
    ### START CODE HERE ###
    expected_output=(
        "Comprehensive research data including: information for each "
        "research topic, and citations used along with source credibility notes"
    ),
    agent=researcher
    ### END CODE HERE ###
)

#define the verify information quality task
verify_information_quality_task = Task(
    description=(
        "Review all collected research. Identify any conflicting information, "
        "potential misinformation, or gaps that need addressing."
    ),
    ### START CODE HERE ###
    expected_output=( 
        "A report with all the original data you got plus any " 
        "verified facts vs. questionable information, make sure this is as comprehensive " 
        "as possible for final report generation"
    ),
    agent=fact_checker
    ### END CODE HERE ###
)

# define the write final report task
write_final_report_task = Task(
    description=(
        "Create a comprehensive report that answers the original query using all verified research data. "
        "Structure it with clear sections, include citations, and provide actionable insights."
    ),
    ### START CODE HERE ###
    expected_output=( 
        "A final research report containing: executive summary, detailed " 
        "findings that answer the user query, supporting evidence and analysis, complete " 
        "source citations."
    ),
    agent=report_writer
    ### END CODE HERE ###
)
    

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [7]:
# create the crew with the defined agents and tasks
crew = Crew(
    ### START CODE HERE ###
    agents=[research_planner, researcher, fact_checker, report_writer],
    tasks=[create_research_plan_task, gather_research_data_task, verify_information_quality_task, write_final_report_task]
    ### 
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [8]:
### START CODE HERE ###

# Write your query, which will be used as input for the tasks.
user_query = None

### END CODE HERE ###

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [9]:
result = crew.kickoff(
    inputs={
        "user_query": "Evaluate the top five emerging AI tools for automating cometitive market analysis including their features, limitations, costs, and ideal uses cases for a mid-sized marketing firm.",
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Based on the user's query, break it down into specific topics and key questions, and create a focused    │
│  research plan.The user's query is: Evaluate the top five emerging AI tools for automating cometitive market    │
│  analysis including their features, limitations, costs, and ideal uses cases for a mid-sized marketing firm.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan for Evaluating Top Five Emerging AI Tools for Automating Competitive Market Analysis**         │
│                                                                                                                 │
│  **Main Research Topics:**                                                                                      │
│                                                                                                                 │
│  1. **Identification of Top Five AI Tools**                                                                     │
│     - **Key Questions:**                                                                                        │
│       - What are the leading emerging AI tools currently available for market analysis?                         │
│       - What criteria should be used to identify and select these tools?                                        │
│       - Which platforms or sources provide rankings or evaluations of such tools?                               │
│     - **Success Criteria:**                                                                                     │
│       - A curated list of the top five emerging AI tools supported by reputable sources.                        │
│       - Clear criteria for evaluation that aligns with the needs of a mid-sized marketing firm.                 │
│                                                                                                                 │
│  2. **Features of Each Tool**                                                                                   │
│     - **Key Questions:**                                                                                        │
│       - What are the primary features of each of the top five AI tools identified?                              │
│       - How do these features contribute to automating competitive market analysis?                             │
│       - Are there any unique functionalities that differentiate the tools from each other?                      │
│     - **Success Criteria:**                                                                                     │
│       - A comparative table outlining the features of each tool.                                                │
│       - Descriptions of key functionalities that are most beneficial for a mid-sized marketing firm.            │
│                                                                                                                 │
│  3. **Limitations of Each Tool**                                                                                │
│     - **Key Questions:**                                                                                        │
│       - What are the known limitations or drawbacks associated with each tool?                                  │
│       - Are there scenarios where these tools may not perform well?                                             │
│       - How do the limitations affect the applicability of each tool for market analysis?                       │
│     - **Success Criteria:**                                                                                     │
│       - A summarized report detailing the limitations and potential pitfalls of each tool.                      │
│       - User reviews or expert opinions that support th

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, collect information on all identified topics. Cite all sources used.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: I need to gather extensive information on the identified topics regarding emerging AI tools for       │
│  automating competitive market analysis. I will start by searching for the top five AI tools currently          │
│  available.                                                                                                     │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "top emerging AI tools for market analysis",                                                 │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: AI Driven Competitive Intelligence Accelerates Research ...                                             │
│  URL:                                                                                                           │
│  https://www.forbes.com/sites/cindygordon/2023/09/26/ai-driven-competitive-intelligence-accelerates-research-p  │
│  roductivity/                                                                                                   │
│  ID:                                                                                                            │
│  https://www.forbes.com/sites/cindygordon/2023/09/26/ai-driven-competitive-intelligence-accelerates-research-p  │
│  roductivity/                                                                                                   │
│  Score: None                                                                                                    │
│  Published Date: 2023-09-26T00:00:00.000Z                                                                       │
│  Author: Cindy Gordon                                                                                           │
│  Image:                                                                                                         │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: AI Driven Competitive Intelligence Accelerates Research Productivity                                     │
│                                                                                                                 │
│  By [Cindy Gordon](https://www.forbes.com/sites/cindygordon/)                                                   │
│                                                                                                                 │
│  Follow Author                                                                                                  │
│                                                                                                                 │
│  Share                                                                                                          │
│                                                                                                                 │
│  SaveComment                                                                                                    │
│                                                                                                                 │
│  [Innovation](https://www.forbes.com/innovation/)[AI](https://www.forbes.com/ai/)                               │
│                                                                                                                 │
│  # AI Driven Competitive Intelligence Accelerates Research Productivity                                         │
│                                                                                                                 │
│  By [Cindy Gordon](https://www.forbes.com/sites/cindygordon/),                                                  │
│                                                                                                                 │
│  Contributor.                                                                                                   │
│                                                                                                                 │
│  Forbes contributors publish independent expert analys

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Checker                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan for Evaluating Top Five Emerging AI Tools for Automating Competitive Market Analysis**         │
│                                                                                                                 │
│  **Main Research Topics:**                                                                                      │
│                                                                                                                 │
│  1. **Identification of Top Five AI Tools**                                                                     │
│     - **Verified Tools**:                                                                                       │
│       - Standard Insights                                                                                       │
│       - Brandwatch                                                                                              │
│       - Quantilope                                                                                              │
│       - Crayon                                                                                                  │
│       - YouScan                                                                                                 │
│       - Additional noted tools include Prelaunch.com, Klue, and AlphaSense.                                     │
│     - **Sources Credited**: AskAttest, HBR, Quantilope, industry articles, and market research. Forbes also     │
│  mentions emerging tools like Prelaunch.com and market leaders Klue and AlphaSense.                             │
│                                                                                                                 │
│  2. **Features of Each Tool**                                                                                   │
│     - **Verified Features**:                                                                                    │
│       | **Tool**          | **Features** |                                                                      │
│       |-------------------|-------------|                                                                       │
│       | Standard Insights | - AI-driven survey creation<br>- Real-time audience sampling<br>- Automated data    │
│  analysis and visualization<br>- Seamless reporting tools |                                                     │
│       | Brandwatch        | - Advanced social listening<br>- Emotion detection<br>- Sentiment analysis<br>-     │
│  Visual content analysis for sentiment |                                                                        │
│       | Quantilope        | - End-to-end research automation<br>- Predictive analytics<br>- Real-time feedback  │
│  gathering |                                                                                                    │
│       | Crayon            | - Competitive intelligence<br>- Real-time monitoring<br>- Event-driven alerts for   │
│  changes in competitor behavior |                                                                               │
│       | YouScan           | - Image and video analysis for social sentiment<br>- Comprehensive insights on      │
│  visual media<br>- Real-time updates from user-generated content |                                              │
│       - Additional Features from Klue and AlphaSense in

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query using all verified research data.          │
│  Structure it with clear sections, include citations, and provide actionable insights.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report on Evaluating Top Five Emerging AI Tools for Automating Competitive Market Analysis     │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  In the fast-evolving market landscape, mid-sized marketing firms are increasingly turning to artificial        │
│  intelligence (AI) tools to enhance their competitive market analysis capabilities. This report identifies and  │
│  evaluates the top five emerging AI tools: Standard Insights, Brandwatch, Quantilope, Crayon, and YouScan. The  │
│  criteria for selecting these tools are based on their features, limitations, cost, and ideal use cases         │
│  tailored for mid-sized marketing firms. This comprehensive evaluation provides actionable insights to help     │
│  firms make informed decisions about which tools may best meet their unique needs.                              │
│                                                                                                                 │
│  ## 1. Tool Identification                                                                                      │
│                                                                                                                 │
│  The following AI tools have been identified as leading options in automating competitive market analysis:      │
│                                                                                                                 │
│  - **Standard Insights**                                                                                        │
│  - **Brandwatch**                                                                                               │
│  - **Quantilope**                                                                                               │
│  - **Crayon**                                                                                                   │
│  - **YouScan**                                                                                                  │
│                                                                                                                 │
│  ### Source Credibility                                                                                         │
│  Information has been compiled from reputable industry reports, articles from platforms like HBR, and           │
│  technology evaluation sources, confirming the position of these tools in the market. For example, AskAttest    │
│  and various expert reviews have discussed their capabilities (AskAttest, 2023; HBR, 2023).                     │
│                                                                                                                 │
│  ## 2. Features of Each Tool                                                                                    │
│                                                                                                                 │
│  A comparative summary outlining the features of each tool is provided below:                                   │
│                                                        

[CrewAIEventsBus] Sync handler error in on_crew_completed: HTTPSConnectionPool(host='app.crewai.com', port=443): 
Max retries exceeded with url: /crewai_plus/api/v1/tracing/batches/d127e9d9-e35d-456e-b34b-278c1b25cb4b (Caused by 
NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D6BDD3B750>: Failed to resolve 
'app.crewai.com' ([Errno 11001] getaddrinfo failed)"))

From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [10]:
from IPython.display import Markdown
Markdown(result.raw) 

**Report on AI Tools for Competitive Market Analysis**

**Executive Summary**  
This report evaluates the top five emerging AI tools designed to automate competitive market analysis, an increasingly critical area for businesses seeking to thrive amid intense competition. By delving into their features, limitations, pricing structures, and ideal use cases, this analysis assists mid-sized marketing firms in making informed decisions on integrating AI technologies into their market strategies. This report presents a comprehensive overview, actionable insights, and detailed comparisons to support strategic implementation.

---

**1. Overview of Emerging AI Tools in Competitive Market Analysis**  

The following tools were identified as the top five emerging AI solutions currently offering advanced features for competitive market analysis:

1. **SEMRush**: A widely recognized tool equipped with features for SEO, PPC optimization, and competitive benchmarking, making it suitable for detailed marketing strategies.
   
2. **Ahrefs**: Known for its backlink analysis capabilities, Ahrefs helps marketers assess their SEO strategies and identify opportunities for improvement by analyzing competitor data.
   
3. **Similarweb**: This tool offers rich insights into web traffic and engagement metrics, allowing businesses to measure their performance against competitors in various digital channels.
   
4. **Visualping**: An AI-driven tool that tracks changes to competitor websites, giving firms real-time notifications of significant updates that may impact market position.
   
5. **AlphaSense**: Focused on financial insights, this tool enables users to conduct advanced searches on news articles and financial documents, extracting meaningful competitive intelligence.

---

**2. Detailed Findings: Features Comparison**  
The following table illustrates the unique features of each identified AI tool:

| Tool         | Key Features                                           | Suitable AI Technology        |
|--------------|-------------------------------------------------------|-------------------------------|
| **SEMRush**  | Keyword analysis, SEO audits, competitive assessments | Data Mining, Machine Learning  |
| **Ahrefs**   | Backlink checking, content gap insights               | Data Analysis, Machine Learning |
| **Similarweb**| Traffic analytics, audience insights                   | Predictive Analytics           |
| **Visualping** | Website monitoring, change alerts                    | Web Scraping, AI Notification  |
| **AlphaSense**| NLP-driven document search, competitive insights      | Natural Language Processing     |

**Supporting Evidence and Analysis**  
- **SEMRush** integrates AI with data analytics to enhance SEO capabilities, making keyword research more efficient than traditional methods.
- **Ahrefs** stands out for its extensive database of backlinks, which enables deep competitive analysis not achievable through manual techniques.
- **Similarweb** utilizes sophisticated algorithms to analyze traffic data, providing insights into user behavior that surpass conventional analytics.
- **Visualping’s** web monitoring provides competitive firms with timely updates, thus allowing immediate strategic adjustments.
- **AlphaSense’s** use of NLP offers a unique advantage in transforming raw data from documents into actionable intelligence—a much-needed leap from traditional manual searches.

---

**3. Limitations of Each AI Tool**  
While each tool presents unique benefits, they also have notable limitations that marketing firms must consider:

- **SEMRush**: High learning curve and potential for escalated costs with add-ons.
- **Ahrefs**: Requires time to learn to utilize its capabilities effectively, particularly for beginners.
- **Similarweb**: Lower-tier subscriptions may lack depth, limiting comprehensive competitive analysis.
- **Visualping**: Primarily focuses on visual changes rather than data-driven insights, which can limit broader analysis.
- **AlphaSense**: High subscription costs can prevent access for smaller or budget-constrained firms.

### Risk-Benefit Matrix
| Limitation                    | Impact on Marketing Firm                     |  
|-------------------------------|----------------------------------------------|  
| High Costs                   | Potential unsustainability for mid-sized firms |  
| Learning Curve               | Increase in onboarding time and training expenses |  
| Limited Data Accessibility    | Restricted strategic options for analysis      |  
| Operational Complexity         | Need for suitable resource allocation for effective use |  

---

**4. Cost Analysis of Each AI Tool**  
Understanding the pricing structures is crucial for budget-conscious firms:

- **SEMRush**: Starts at $117.33/month with tiered options.
- **Ahrefs**: Priced at around $129/month, provides discounts for annual subscriptions.
- **Similarweb**: Basic plans begin at $199/month; custom pricing for advanced features.
- **Visualping**: Entry pricing starts at $13/month; higher tiers offer enhanced functionalities.
- **AlphaSense**: Pricing is customized based on usage; can be premium-cost prohibitive.

### Potential Return on Investment
Investing in these tools could lead to enhanced market insights, optimized marketing strategies, and potentially increased revenue based on informed decision-making.

---

**5. Ideal Use Cases for a Mid-Sized Marketing Firm**  
Here are specific scenarios tailored for each tool, illustrating how they can integrate into a marketing firm’s workflow:

- **SEMRush**: Ideal for launching a competitive PPC campaign by evaluating competitors’ ad strategies and keyword utilization.
- **Ahrefs**: Best for assessing current SEO performance and determining missing content opportunities through competitor links analysis.
- **Similarweb**: Useful for unveiling audience behavior patterns, leading to refined targeting in marketing efforts.
- **Visualping**: Effective for monitoring changes on competitor websites to adapt quickly and maintain a competitive edge.
- **AlphaSense**: Beneficial for keeping abreast of market shifts via financial document insights, aiding in strategic planning.

---

**Conclusion**  
The integration of AI tools into competitive market analysis presents opportunities for mid-sized firms to gain insights that drive strategic decisions. However, firms must navigate decisions carefully regarding tool selection, weighing features against potential limits and costs. This structured exploration of the top tools listed provides foundational information necessary for contemporary marketing firms to thrive in a competitive marketplace effectively.

---

**Sources**
1. Competitive Intelligence Alliance - "5 key limitations of generative AI in competitive intelligence." [Link](https://www.competitiveintelligencealliance.io/5-key-limitations-of-generative-ai-in-competitive-intelligence/)
2. CODESM - "AI for Competitive Analysis How to Outrank and Outperform Rivals." [Link](https://www.codesm.com/blog/ai-automate-competitive-analysis/)
3. Quantilope - "10 AI Market Research Tools & How To Use Them." [Link](https://www.quantilope.com/resources/best-ai-market-research-tools)
4. Harvard DCE Blog - "AI Will Shape the Future of Marketing." [Link](https://professional.dce.harvard.edu/blog/ai-will-shape-the-future-of-marketing/)

With this comprehensive report, mid-sized marketing firms are better equipped to choose the tools that align best with their operational needs and strategic goals for market analysis.

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!